<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICS40125 - Laboratorio N°04


**Objetivo**: Aplicar técnicas intermedias y avanzadas de análisis de datos con pandas utilizando un caso real: el Índice de Libertad de Prensa. Este laboratorio incluye operaciones de limpieza, transformación, combinación de datos, y análisis exploratorio usando `merge`, `groupby`, `concat` y otras funciones fundamentales.




**Descripción del Dataset**

El presente conjunto de datos está orientado al análisis del **Índice de Libertad de Prensa**, una métrica internacional que evalúa el nivel de libertad del que gozan periodistas y medios de comunicación en distintos países. Este índice es recopilado anualmente por la organización **Reporteros sin Fronteras**.

La base de datos contempla observaciones por país y año, e incluye tanto el valor del índice como el ranking correspondiente. A menor puntaje en el índice, mayor nivel de libertad de prensa.

**Diccionario de variables**

| Variable     | Clase    | Descripción                                                                          |
| ------------ | -------- | ------------------------------------------------------------------------------------ |
| `codigo_iso` | carácter | Código ISO 3166-1 alfa-3 que representa a cada país.                                 |
| `pais`       | carácter | Nombre oficial del país.                                                             |
| `anio`       | entero   | Año en que se registró la medición del índice.                                       |
| `indice`     | numérico | Valor numérico del Índice de Libertad de Prensa (menor valor indica mayor libertad). |
| `ranking`    | entero   | Posición relativa del país en el ranking mundial de libertad de prensa.              |


**Fuente original y adaptación pedagógica**

* **Fuente original**: [Reporteros sin Fronteras](https://www.rsf-es.org/), recopilado y publicado a través del portal del [Banco Mundial](https://tcdata360.worldbank.org/indicators/h3f86901f?country=BRA&indicator=32416&viz=line_chart&years=2001,2019).
* **Adaptación educativa**: Los archivos han sido modificados intencionalmente para incorporar desafíos técnicos que permiten aplicar los contenidos abordados en clases, tales como limpieza de datos, normalización, detección de duplicados, y combinación de fuentes.


**Descripción de los archivos disponibles**

* **`libertad_prensa_codigo.csv`**: Contiene los pares `codigo_iso` y `pais`. Incluye intencionalmente un código ISO con dos nombres distintos de país para efectos de limpieza y validación de datos.

* **`libertad_prensa_01.csv`**: Contiene registros de los años **anteriores a 2010**. Incluye las variables `PAIS`, `ANIO`, `INDICE`, y `RANKING` con nombres de columna en **mayúsculas**.

* **`libertad_prensa_02.csv`**: Contiene registros de los años **desde 2010 en adelante**. Estructura similar al archivo anterior, con nombres de columna también en **mayúsculas**.





In [1]:
import numpy as np
import pandas as pd

# lectura de datos
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_02.csv'
 ]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_codigo.csv')



### 1. Consolidación y limpieza de datos

A partir de los archivos disponibles, realice los siguientes pasos:

**a)** Cree un DataFrame llamado `df_anio` que consolide la información proveniente de los archivos **`libertad_prensa_01.csv`** y **`libertad_prensa_02.csv`**, correspondientes a distintas ventanas de tiempo. Recuerde que ambos archivos tienen nombres de columnas en mayúscula, por lo que debe normalizarlas a **minúscula** para asegurar consistencia.

**b)** Explore el archivo **`libertad_prensa_codigo.csv`** e identifique el código ISO que aparece asociado a dos nombres de país distintos. Elimine el registro que corresponda a un valor incorrecto o inconsistente, conservando solo el que considere válido.

**c)** Una vez preparados los archivos, cree un nuevo DataFrame llamado `df` que combine `df_anio` con `df_codigos`, utilizando la columna `codigo_iso` como clave. Asegúrese de realizar una unión que conserve únicamente los registros que tengan coincidencia en ambas fuentes.

> **Sugerencia**:
>
> * Para unir los archivos por filas (años), utilice la función `pd.concat([...])`.
> * Para combinar información por columnas (variables), utilice `pd.merge(...)` especificando `on='codigo_iso'`.



In [2]:
dfs_to_concat = []
for url in archivos_anio:
    df_temp = pd.read_csv(url)
    df_temp.columns = df_temp.columns.str.lower()  # Normalize column names to lowercase
    dfs_to_concat.append(df_temp)

df_anio = pd.concat(dfs_to_concat, ignore_index=True)

# Identify and remove the inconsistent ISO code/country pair
# The description mentions 'ZWE' with 'malo' as an inconsistent entry.
# First, check for `codigo_iso` that appear with more than one `pais`.
duplicate_iso_check = df_codigos.groupby('codigo_iso')['pais'].nunique()
inconsistent_iso = duplicate_iso_check[duplicate_iso_check > 1].index.tolist()

if inconsistent_iso:
    for iso in inconsistent_iso:
        # Assuming 'malo' is the incorrect entry as indicated in the description
        df_codigos = df_codigos[~((df_codigos['codigo_iso'] == iso) & (df_codigos['pais'] == 'malo'))]

# Create df by merging df_anio with df_codigos
df = pd.merge(df_anio, df_codigos, on='codigo_iso', how='inner')

display(df.head())

,codigo_iso,anio,indice,ranking,pais
0,AFG,2001,35.5,59.0,Afghanistán
1,AGO,2001,30.2,50.0,Angola
2,ALB,2001,NaN,NaN,Albania
3,AND,2001,NaN,NaN,Andorra
4,ARE,2001,NaN,NaN,Emiratos Árabes Unidos




### 2. Exploración inicial del conjunto de datos

Una vez que hayas consolidado el DataFrame final `df`, realiza un análisis exploratorio básico respondiendo las siguientes preguntas:

#### **Estructura del DataFrame**

* ¿Cuántas **filas (observaciones)** contiene el conjunto de datos?
* ¿Cuántas **columnas** tiene el DataFrame?
* ¿Cuáles son los **nombres de las columnas**?
* ¿Qué **tipo de datos** tiene cada columna?
* ¿Hay columnas con un tipo de dato inesperado (por ejemplo, fechas como strings)?

#### **Resumen estadístico**

* Genera un resumen estadístico del conjunto de datos con `.describe()`.
  ¿Qué observas sobre los valores de `indice` y `ranking`?
* ¿Qué valores mínimo, máximo y promedio tiene la columna `indice`?
* ¿Qué países presentan los valores extremos en `indice` y `ranking`?

#### **Datos faltantes**

* ¿Cuántos valores nulos hay en cada columna?
* ¿Qué proporción de observaciones tienen valores faltantes?
* ¿Hay columnas con más del 30% de datos faltantes?

#### **Unicidad y duplicados**

* ¿Cuántos países distintos (`pais`) hay en el DataFrame?
* ¿Cuántos años distintos (`anio`) hay representados?
* ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?

#### **Validación cruzada de columnas**

* ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
  (por ejemplo, un mismo código ISO asociado a más de un país)

> **Sugerencia**: Apoya tu análisis con funciones como `.info()`, `.nunique()`, `.isnull().sum()`, `.duplicated()`, `.value_counts()`, entre otras.



    

## **Estructura del Dataframe**

In [7]:
# Number of rows and columns
print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

# Column names
print("Nombres de las columnas:", df.columns.tolist())

# Data types of each column and check for unexpected types
print("\nInformación general del DataFrame:")
df.info()

Número de filas: 3060
Número de columnas: 5
Nombres de las columnas: ['codigo_iso', 'anio', 'indice', 'ranking', 'pais']

Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3060 entries, 0 to 3059
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   codigo_iso  3060 non-null   object 
 1   anio        3060 non-null   int64  
 2   indice      2664 non-null   float64
 3   ranking     2837 non-null   float64
 4   pais        3060 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 119.7+ KB


**Número de filas (observaciones):** El DataFrame contiene 3060 filas.

**Número de columnas:** El DataFrame tiene 5 columnas.

**Nombres de las columnas:** Las columnas son: ['codigo_iso', 'anio', 'indice', 'ranking', 'pais'].

**Tipo de datos por columna:**
- codigo_iso: object (cadena de texto)
- anio: int64 (entero)
- indice: float64 (número decimal)
- ranking: float64 (número decimal)
- pais: object (cadena de texto)

**Tipos de datos inesperados:** No hay columnas con tipos de datos inesperados. Los tipos object para cadenas y int64/float64 para números son apropiados.

## **Resumen estadístico**

In [8]:
# Generar resumen estadístico
display(df.describe())

,anio,indice,ranking
count,3060.000000,2664.000000,2837.000000
mean,2009.941176,205.782316,477.930913
std,5.786024,2695.525264,6474.935347
min,2001.000000,0.000000,1.000000
25%,2005.000000,15.295000,34.000000
50%,2009.000000,28.000000,70.000000
75%,2015.000000,41.227500,110.000000
max,2019.000000,64536.000000,121056.000000


**ResumenEstadístico (mediante .describe()):**

- anio: Los años de observación van desde 2001 hasta 2019, con un promedio alrededor de 2009.9.

- indice: Presenta un valor mínimo de 0.0 y un máximo de 64536.0, con una media de 205.78. La desviación estándar es muy alta (2695.52), lo que indica una gran variabilidad en los datos. Esto sugiere la presencia de valores atípicos muy grandes o un rango de valores extremadamente amplio. El 75% de los valores están por debajo de 41.23, lo que hace que el máximo de 64536.0 sea particularmente notable.

- ranking: El valor mínimo es 1.0 y el máximo es 121056.0, con una media de 477.93. También muestra una gran desviación estándar (6474.93), similar al indice, lo que indica valores atípicos o una escala de ranking muy variable.

**Valores mínimo, máximo y promedio de la columna indice:**

- Valor mínimo de indice: 0.0

- Valor máximo de indice: 64536.0

- Valor promedio de indice: 205.78

In [12]:
# País con el índice más bajo (mayor libertad de prensa)
country_min_indice = df.loc[df['indice'].idxmin()]
print("País con el menor 'indice' (mayor libertad de prensa):")
display(country_min_indice[['pais', 'anio', 'indice']])

# País con el índice más alto (menor libertad de prensa)
country_max_indice = df.loc[df['indice'].idxmax()]
print("\nPaís con el mayor 'indice' (menor libertad de prensa):")
display(country_max_indice[['pais', 'anio', 'indice']])

# País con el ranking más bajo (mejor posición)
country_min_ranking = df.loc[df['ranking'].idxmin()]
print("\nPaís con el menor 'ranking' (mejor posición):")
display(country_min_ranking[['pais', 'anio', 'ranking']])

# País con el ranking más alto (peor posición)
country_max_ranking = df.loc[df['ranking'].idxmax()]
print("\nPaís con el mayor 'ranking' (peor posición):")
display(country_max_ranking[['pais', 'anio', 'ranking']])

País con el menor 'indice' (mayor libertad de prensa):


,1304
pais,Dinamarca
anio,2008
indice,0.0



País con el mayor 'indice' (menor libertad de prensa):


,2069
pais,Kosovo
anio,2014
indice,64536.0



País con el menor 'ranking' (mejor posición):


,53
pais,Finlandia
anio,2001
ranking,1.0



País con el mayor 'ranking' (peor posición):


,2249
pais,Kosovo
anio,2015
ranking,121056.0


**Países con valores extremos en indice y ranking:**

- País con el menor indice (mayor libertad de prensa):
País: Dinamarca
Año: 2008
Índice: 0.0

- País con el mayor indice (menor libertad de prensa):
País: Kosovo
Año: 2014
Índice: 64536.0

- País con el menor ranking (mejor posición):
País: Finlandia
Año: 2001
Ranking: 1.0

- País con el mayor ranking (peor posición):
País: Kosovo
Año: 2015
Ranking: 121056.0

### **Datos faltantes**

In [14]:
# Contar valores nulos por columna
null_counts = df.isnull().sum()
print("Número de valores nulos por columna:")
display(null_counts)

# Proporción de observaciones con valores faltantes
missing_rows = df.isnull().any(axis=1).sum()
total_rows = len(df)
proportion_missing_rows = (missing_rows / total_rows) * 100
print(f"\nProporción de filas con al menos un valor faltante: {proportion_missing_rows:.2f}%\n")

# Proporción de valores faltantes por columna
null_proportions = (df.isnull().sum() / len(df)) * 100
print("Proporción de valores nulos por columna (%):")
display(null_proportions)


Número de valores nulos por columna:


,0
codigo_iso,0
anio,0
indice,396
ranking,223
pais,0



Proporción de filas con al menos un valor faltante: 12.97%

Proporción de valores nulos por columna (%):


,0
codigo_iso,0.000000
anio,0.000000
indice,12.941176
ranking,7.287582
pais,0.000000


- Número de valores nulos por columna:

codigo_iso: 0

anio: 0

indice: 396 valores nulos

ranking: 223 valores nulos

pais: 0

- Proporción de observaciones con valores faltantes:
La proporción de filas con al menos un valor faltante es del 12.97%.
Específicamente, el indice tiene un 12.94% de valores nulos y el ranking tiene un 7.29% de valores nulos.
Columnas con más del 30% de datos faltantes:

- No hay ninguna columna que tenga más del 30% de datos faltantes.

### **Unicidad y duplicados**

In [15]:
# ¿Cuántos países distintos (`pais`) hay en el DataFrame?
num_distinct_countries = df['pais'].nunique()
print(f"Número de países distintos: {num_distinct_countries}")

# ¿Cuántos años distintos (`anio`) hay representados?
num_distinct_years = df['anio'].nunique()
print(f"Número de años distintos: {num_distinct_years}")

# ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?
num_duplicate_rows = df.duplicated().sum()
print(f"Número de filas duplicadas: {num_duplicate_rows}")

Número de países distintos: 179
Número de años distintos: 17
Número de filas duplicadas: 0


- Hay **179** países distintos representados en el DataFrame.

- Se registran datos para **17** años distintos.

- **No se encontraron** filas duplicadas en el DataFrame.


### **Validación cruzada de columnas**

In [16]:
# ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
# (por ejemplo, un mismo código ISO asociado a más de un país)

inconsistent_country_iso = df.groupby('codigo_iso')['pais'].nunique()
inconsistent_country_iso = inconsistent_country_iso[inconsistent_country_iso > 1]

if not inconsistent_country_iso.empty:
    print("Se encontraron inconsistencias entre 'pais' y 'codigo_iso':")
    for iso in inconsistent_country_iso.index:
        countries = df[df['codigo_iso'] == iso]['pais'].unique().tolist()
        print(f"  Código ISO '{iso}' está asociado a los países: {', '.join(countries)}")
else:
    print("No se encontraron inconsistencias entre 'pais' y 'codigo_iso'.")

No se encontraron inconsistencias entre 'pais' y 'codigo_iso'.


Validación Cruzada de Columnas:
- No se encontraron inconsistencias entre el país (pais) y su código (codigo_iso), lo que significa que cada codigo_iso está asociado a un único pais.




### 3. Comparación regional: países latinoamericanos

En esta sección se busca identificar cuáles son los países de América Latina que han presentado los valores extremos del **Índice de Libertad de Prensa** en cada año observado.

> Recuerda que un menor puntaje en `indice` implica mayor libertad de prensa.

#### **Tareas:**

**a)** Utilizando un ciclo `for`, recorre cada año del conjunto de datos filtrado por países latinoamericanos, y determina para cada año:

* El país con el menor valor de `indice` (mayor libertad de prensa).
* El país con el mayor valor de `indice` (menor libertad de prensa).

**b)** Resuelve la misma tarea del punto anterior utilizando un enfoque vectorizado con `groupby`, sin usar ciclos explícitos.



#### **Lista de países latinoamericanos considerada:**

```python
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']
```

> Puedes usar esta lista para filtrar el DataFrame final por la columna `codigo_iso`.



In [20]:
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']

df_america = df[df['codigo_iso'].isin(america)].copy()

print("DataFrame de países latinoamericanos (df_america) creado. Primeras 5 filas:")
display(df_america.head())

DataFrame de países latinoamericanos (df_america) creado. Primeras 5 filas:


,codigo_iso,anio,indice,ranking,pais
5,ARG,2001,12.0,8.0,Argentina
7,ATG,2001,NaN,NaN,Antigua y Barbuda
20,BLZ,2001,NaN,NaN,Belize
21,BOL,2001,14.5,13.0,Bolivia
22,BRA,2001,18.8,18.0,Brasil


#### **a) Encontrar valores extremos de `indice` por año usando un ciclo `for`**

In [21]:
print("\nAnálisis de valores extremos del índice por año (usando bucle for):\n")

years = df_america['anio'].unique()

for year in sorted(years):
    df_year = df_america[df_america['anio'] == year]

    # Filtrar NaN en 'indice' para evitar errores con idxmin/idxmax
    df_year_filtered = df_year.dropna(subset=['indice'])

    if not df_year_filtered.empty:
        # País con el menor índice (mayor libertad de prensa)
        min_indice_country = df_year_filtered.loc[df_year_filtered['indice'].idxmin()]

        # País con el mayor índice (menor libertad de prensa)
        max_indice_country = df_year_filtered.loc[df_year_filtered['indice'].idxmax()]

        print(f"--- Año {year} ---")
        print(f"  Mayor libertad de prensa: {min_indice_country['pais']} (Indice: {min_indice_country['indice']:.2f})")
        print(f"  Menor libertad de prensa: {max_indice_country['pais']} (Indice: {max_indice_country['indice']:.2f})")
    else:
        print(f"--- Año {year} ---")
        print(f"  No hay datos de 'indice' disponibles para este año.")


Análisis de valores extremos del índice por año (usando bucle for):

--- Año 2001 ---
  Mayor libertad de prensa: Canadá (Indice: 0.80)
  Menor libertad de prensa: Cuba (Indice: 90.30)
--- Año 2002 ---
  Mayor libertad de prensa: Trinidad y Tobago (Indice: 1.00)
  Menor libertad de prensa: Cuba (Indice: 97.83)
--- Año 2003 ---
  Mayor libertad de prensa: Trinidad y Tobago (Indice: 2.00)
  Menor libertad de prensa: Argentina (Indice: 35826.00)
--- Año 2004 ---
  Mayor libertad de prensa: Trinidad y Tobago (Indice: 2.00)
  Menor libertad de prensa: Cuba (Indice: 87.00)
--- Año 2005 ---
  Mayor libertad de prensa: Bolivia (Indice: 4.50)
  Menor libertad de prensa: Cuba (Indice: 95.00)
--- Año 2006 ---
  Mayor libertad de prensa: Canadá (Indice: 4.88)
  Menor libertad de prensa: Cuba (Indice: 96.17)
--- Año 2007 ---
  Mayor libertad de prensa: Canadá (Indice: 3.33)
  Menor libertad de prensa: Cuba (Indice: 88.33)
--- Año 2008 ---
  Mayor libertad de prensa: Canadá (Indice: 3.70)
  Menor l

#### **b) Encontrar valores extremos de `indice` por año usando `groupby` (enfoque vectorizado)**

In [22]:
print("\nAnálisis de valores extremos del índice por año (usando groupby):\n")

# Filtrar las filas donde 'indice' no es NaN antes de agrupar
df_america_clean = df_america.dropna(subset=['indice'])

# Encontrar el país con el menor índice por año
idx_min = df_america_clean.groupby('anio')['indice'].idxmin()
min_indice_countries_df = df_america_clean.loc[idx_min.dropna(), ['anio', 'pais', 'indice']]
min_indice_countries_df.rename(columns={'pais': 'Pais_Mayor_Libertad', 'indice': 'Indice_Mayor_Libertad'}, inplace=True)

# Encontrar el país con el mayor índice por año
idx_max = df_america_clean.groupby('anio')['indice'].idxmax()
max_indice_countries_df = df_america_clean.loc[idx_max.dropna(), ['anio', 'pais', 'indice']]
max_indice_countries_df.rename(columns={'pais': 'Pais_Menor_Libertad', 'indice': 'Indice_Menor_Libertad'}, inplace=True)

# Combinar los resultados para una mejor visualización
annual_extremes = pd.merge(min_indice_countries_df, max_indice_countries_df, on='anio', suffixes=('_min', '_max'))
annual_extremes.set_index('anio', inplace=True)
display(annual_extremes)


Análisis de valores extremos del índice por año (usando groupby):



,Pais_Mayor_Libertad,Indice_Mayor_Libertad,Pais_Menor_Libertad,Indice_Menor_Libertad
anio,,,,
2001,Canadá,0.80,Cuba,90.30
2002,Trinidad y Tobago,1.00,Cuba,97.83
2003,Trinidad y Tobago,2.00,Argentina,35826.00
2004,Trinidad y Tobago,2.00,Cuba,87.00
2005,Bolivia,4.50,Cuba,95.00
2006,Canadá,4.88,Cuba,96.17
2007,Canadá,3.33,Cuba,88.33
2008,Canadá,3.70,Cuba,94.00
2009,Estados Unidos,6.75,Cuba,78.00


### 4. Análisis anual del índice por país

En esta sección se busca analizar la evolución del **índice máximo** de libertad de prensa alcanzado por cada país a lo largo del tiempo.

#### **Tarea principal:**

* Construye una tabla dinámica (`pivot_table`) donde las **filas** correspondan a los países, las **columnas** a los años (`anio`) y los **valores** sean el `indice` máximo alcanzado por cada país en ese año.
* Asegúrate de reemplazar los valores nulos resultantes con `0`.

> **Hint**: Puedes utilizar el parámetro `fill_value=0` en `pd.pivot_table(...)`.



#### **Preguntas adicionales:**

**a)** ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
**b)** ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?

> (Pista: usa `.mean(axis=0)` sobre la tabla pivot)

**c)** ¿Qué país muestra mayor **variabilidad** (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?

> (Pista: aplica `.max(axis=1) - .min(axis=1)`)

**d)** ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?

**e)** ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?





### **4. Análisis anual del índice por país**

In [23]:
# Construye una tabla dinámica (pivot_table)
pivot_table = df_america.pivot_table(index='pais', columns='anio', values='indice', aggfunc='max', fill_value=0)
display(pivot_table.head())

anio,2001,2002,2003,2004,2005,2006,2007,2008,2009,2012,2013,2014,2015,2017,2018,2019
pais,,,,,,,,,,,,,,,,
Antigua y Barbuda,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,20.81,0.00,0.00,0.00,0.00,0.00
Argentina,12.0,15.17,35826.0,13.67,17.30,24.83,14.08,11.33,16.35,25.67,25.27,26.11,25.09,25.07,26.05,28.30
Belize,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,17.05,18.54,20.61,23.43,24.55,27.50
Bolivia,14.5,9.67,20.0,9.67,4.50,21.50,28.20,24.17,28.13,32.80,31.04,31.29,31.78,33.88,32.45,35.38
Brasil,18.8,16.75,16.5,14.50,17.17,25.25,18.00,15.88,16.60,32.75,34.03,31.93,32.62,33.58,31.20,32.79


#### **a) ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?**

In [24]:
max_indice_overall = pivot_table.max().max()
country_max_indice_overall = pivot_table[pivot_table == max_indice_overall].stack().index[0][0]

min_indice_overall = pivot_table[pivot_table > 0].min().min()
country_min_indice_overall = pivot_table[pivot_table == min_indice_overall].stack().index[0][0]

print(f"País con el mayor valor de índice en toda la tabla: {country_max_indice_overall} (Índice: {max_indice_overall:.2f})")
print(f"País con el menor valor de índice (distinto de cero) en toda la tabla: {country_min_indice_overall} (Índice: {min_indice_overall:.2f})")

País con el mayor valor de índice en toda la tabla: Argentina (Índice: 35826.00)
País con el menor valor de índice (distinto de cero) en toda la tabla: Canadá (Índice: 0.80)


#### **b) ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?**

In [25]:
average_indice_by_year = pivot_table.mean(axis=0)

year_highest_avg_indice = average_indice_by_year.idxmax()
value_highest_avg_indice = average_indice_by_year.max()

year_lowest_avg_indice = average_indice_by_year.idxmin()
value_lowest_avg_indice = average_indice_by_year.min()

print(f"Año con el promedio de índice más alto: {year_highest_avg_indice} (Promedio: {value_highest_avg_indice:.2f})")
print(f"Año con el promedio de índice más bajo: {year_lowest_avg_indice} (Promedio: {value_lowest_avg_indice:.2f})")

Año con el promedio de índice más alto: 2003 (Promedio: 1251.68)
Año con el promedio de índice más bajo: 2001 (Promedio: 12.42)


#### **c) ¿Qué país muestra mayor variabilidad (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?**

In [26]:
variability_by_country = (pivot_table.max(axis=1) - pivot_table[pivot_table > 0].min(axis=1)).dropna()
country_most_variability = variability_by_country.idxmax()
value_most_variability = variability_by_country.max()

print(f"País con mayor variabilidad en el índice: {country_most_variability} (Diferencia: {value_most_variability:.2f})")

País con mayor variabilidad en el índice: Argentina (Diferencia: 35814.67)


#### **d) ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?**

In [27]:
constant_indice_countries = []
for index, row in pivot_table.iterrows():
    # Consider only years with actual data (indice > 0)
    valid_indices = row[row > 0]
    if not valid_indices.empty and valid_indices.nunique() == 1:
        constant_indice_countries.append(index)

if constant_indice_countries:
    print(f"Países con índice constante (distinto de cero) a lo largo de los años: {', '.join(constant_indice_countries)}")
else:
    print("No se encontraron países con índice constante a lo largo de los años registrados (distinto de cero).")

Países con índice constante (distinto de cero) a lo largo de los años: Antigua y Barbuda, Granada


#### **e) ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?**

In [28]:
countries_all_zeros = pivot_table[(pivot_table == 0).all(axis=1)].index.tolist()

if countries_all_zeros:
    print(f"Países sin datos (todos los valores igual a 0) en la tabla dinámica: {', '.join(countries_all_zeros)}")
    print("Esto se debe a que, para estos países y años específicos en el DataFrame `df_america`, no existían valores de 'indice' y fueron rellenados con 0 durante la creación de la tabla dinámica con `fill_value=0`.")
else:
    print("No se encontraron países con todos los valores igual a 0 en la tabla dinámica.")

No se encontraron países con todos los valores igual a 0 en la tabla dinámica.


In [ ]:
# FIX ME